# Word2Vec Models

Train and persist the Word2Vec-based spam classifiers and embedding model.

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import preprocess_many

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'SMSSpamCollection.txt'
MODEL_RF_PATH = PROJECT_ROOT / 'models' / 'word2vec_rf.pkl'
MODEL_XGB_PATH = PROJECT_ROOT / 'models' / 'avg_word2vec_xgb.pkl'
WORD2VEC_PATH = PROJECT_ROOT / 'vectorizers' / 'word2vec.model'

messages = pd.read_csv(DATA_PATH, sep='\t', names=['label', 'message'])
corpus = preprocess_many(messages['message'].astype(str).tolist())
sentences = [doc.split() for doc in corpus]
y = messages['label'].map({'ham': 0, 'spam': 1}).to_numpy()
X_train_text, X_test_text, y_train, y_test = train_test_split(sentences, y, test_size=0.20, random_state=42, stratify=y)
w2v = Word2Vec(sentences=X_train_text, vector_size=100, window=5, min_count=1, workers=2, sg=1, epochs=50)
WORD2VEC_PATH.parent.mkdir(parents=True, exist_ok=True)
w2v.save(str(WORD2VEC_PATH))


def sentence_vector(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


X_train_vec = np.vstack([sentence_vector(tokens, w2v) for tokens in X_train_text])
X_test_vec = np.vstack([sentence_vector(tokens, w2v) for tokens in X_test_text])
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_vec, y_train)
rf_pred = rf_model.predict(X_test_vec)
print('RandomForest accuracy:', accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    eval_metric='logloss',
)
xgb_model.fit(X_train_vec, y_train)
xgb_pred = xgb_model.predict(X_test_vec)
print('XGBoost accuracy:', accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))
MODEL_RF_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_XGB_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_model, MODEL_RF_PATH)
joblib.dump(xgb_model, MODEL_XGB_PATH)
print(MODEL_RF_PATH)
print(MODEL_XGB_PATH)
print(WORD2VEC_PATH)

RandomForest accuracy: 0.9730941704035875
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.99      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115

XGBoost accuracy: 0.9820627802690582
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       966
           1       0.96      0.90      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

c:\Users\Yashesh Mehta\Desktop\Coding\SpamShield AI NLP-based Spam Detection System\models\word2vec_rf.pkl
c:\Users\Yashesh Mehta\Desktop\Coding\SpamShield AI NLP-based Spam Detection System\models\avg_word2vec_xgb.pkl
c:\Users\Yashesh Mehta\Desktop\Coding\SpamShield